In [31]:
import pandas as pd
import os 
#open the csv file with the tidy_all_data dataframe
wellbeing_communities = pd.read_csv(os.path.join('..','..','data','processed','matched_cwb_2021.csv'), usecols=['CSD Name 2021'])
#tidy_all_data = pd.read_csv('../../data/processed/tidy_all_metrics.csv')

In [32]:
#get coordinates of each city in the bc-gazetteer-2024-06-13.csv file in the raw data folder
gazeteer = pd.read_csv(os.path.join('..','..','data','raw','bc-gazetteer-2024-06-13.csv'))
# Filter for communities in the final analysis data
gazeteer_filtered = gazeteer[gazeteer['Official Name'].isin(wellbeing_communities['CSD Name 2021']) & gazeteer['Feature Type'].isin(['City', 'Town', 'District Municipality', 'District Municipality (1)', 'Village (1)','Resort Municipality', 'Mountain Resort Municipality'])]


In [33]:
#print all unique official names in the gazeteer_filtered dataframe
print("Unique official names in gazeteer_filtered:")
print(gazeteer_filtered['Official Name'].unique())
#count unique official names
print(f"Total unique official names in gazeteer_filtered: {gazeteer_filtered['Official Name'].nunique()}")
#figure out which communities are missing coordinates
missing_coordinates = wellbeing_communities[~wellbeing_communities['CSD Name 2021'].isin(gazeteer_filtered['Official Name'])]
print(f"Communities missing coordinates: {len(missing_coordinates)}")
print("Missing communities:")
for community in missing_coordinates['CSD Name 2021'].unique():
    print(f"  {community}")

Unique official names in gazeteer_filtered:
['Abbotsford' 'Alert Bay' 'Anmore' 'Armstrong' 'Ashcroft' 'Belcarra'
 'Bowen Island' 'Burnaby' 'Burns Lake' 'Cache Creek' 'Campbell River'
 'Canal Flats' 'Castlegar' 'Central Saanich' 'Chase' 'Chetwynd'
 'Chilliwack' 'Clinton' 'Coldstream' 'Colwood' 'Coquitlam' 'Courtenay'
 'Cranbrook' 'Creston' 'Cumberland' 'Dawson Creek' 'Delta' 'Duncan'
 'Enderby' 'Esquimalt' 'Fernie' 'Fort St. James' 'Fort St. John'
 'Fraser Lake' 'Fruitvale' 'Gibsons' 'Gold River' 'Golden' 'Grand Forks'
 'Granisle' 'Greenwood' 'Harrison Hot Springs' 'Highlands' 'Hope'
 'Houston' "Hudson's Hope" 'Invermere' 'Kamloops' 'Kaslo' 'Kelowna' 'Kent'
 'Keremeos' 'Kimberley' 'Kitimat' 'Ladysmith' 'Lake Country'
 'Lake Cowichan' 'Langford' 'Lantzville' 'Lillooet' 'Lions Bay'
 'Logan Lake' 'Lumby' 'Mackenzie' 'Maple Ridge' 'Masset' 'McBride'
 'Merritt' 'Metchosin' 'Midway' 'Mission' 'Montrose' 'Nakusp' 'Nanaimo'
 'Nelson' 'New Denver' 'New Hazelton' 'New Westminster' 'North Cowichan

In [34]:
# Manual mapping for missing communities using both name and feature type
name_featuretype_corrections = {
    "Langley City": ("Langley", "City"),
    "Langley Township": ("Langley", "District Municipality (1)"),
    "North Vancouver City": ("North Vancouver", "City"),
    "North Vancouver District": ("North Vancouver", "District Municipality (1)"),
    "One Hundred Mile House": ("100 Mile House", "District Municipality (1)"),
    "Sun Peaks Mountain Resort": ("Sun Peaks", "Mountain Resort Municipality"),
    "West Kelowna": ("West Kelowna, City of", "City"),
    "Queen Charlotte": ("Queen Charlotte", "Post Office"),
    "Sechelt IGD": ("Sechelt Indian Government District", "Indian Government District"),
    "Northern Rockies" :("Northern Rockies Regional Municipality","District Municipality (1)") 
}

missing_coords = []
for community, (gaz_name, feature_type) in name_featuretype_corrections.items():
    if feature_type:
        match = gazeteer[(gazeteer['Official Name'].str.lower() == gaz_name.lower()) &
                         (gazeteer['Feature Type'].str.lower() == feature_type.lower())]
    else:
        match = gazeteer[gazeteer['Official Name'].str.lower() == gaz_name.lower()]
    if not match.empty:
        feature_type = match.iloc[0]['Feature Type']
        lat = match.iloc[0]['LatDD']
        lon = match.iloc[0]['LongDD']
        datum = match.iloc[0]['Datum']
        missing_coords.append({'Official Name': community, 'Feature Type': feature_type, 'Datum': datum, 'LatDD': lat, 'LongDD': lon})
    else:
        print(f"Not found in gazeteer: {community} -> {gaz_name} ({feature_type})")

missing_coords_df = pd.DataFrame(missing_coords)
print(missing_coords_df)

               Official Name                  Feature Type  Datum      LatDD  \
0               Langley City                          City  WGS84  49.102500   
1           Langley Township     District Municipality (1)  WGS84  49.120278   
2       North Vancouver City                          City  WGS84  49.320556   
3   North Vancouver District     District Municipality (1)  WGS84  49.336111   
4     One Hundred Mile House     District Municipality (1)  WGS84  51.642778   
5  Sun Peaks Mountain Resort  Mountain Resort Municipality  WGS84  50.884444   
6               West Kelowna                          City  WGS84  49.830000   
7            Queen Charlotte                   Post Office  WGS84  53.253878   
8                Sechelt IGD    Indian Government District  WGS84  49.478960   
9           Northern Rockies     District Municipality (1)  WGS84  58.803889   

       LongDD  
0 -122.658056  
1 -122.659722  
2 -123.073889  
3 -123.078056  
4 -121.295556  
5 -119.879444  
6 -119.

In [35]:
#add missing communities to gazeteer_filtered
gazeteer_filtered = pd.concat([gazeteer_filtered, missing_coords_df]).reset_index(drop=True)
gazeteer_filtered.head()
#gazeteer_filtered.tail(20)

,Official Name,Feature Type,Feature Type Code,Mapsheet,Latitude,Longitude,Datum,LatDD,LongDD
0,Abbotsford,City,1.0,92G/1,49 03 07 N,122 19 45 W,WGS84,49.052222,-122.329167
1,Alert Bay,Village (1),3.0,92L/10,50 35 01 N,126 55 40 W,WGS84,50.583889,-126.927778
2,Anmore,Village (1),3.0,92G/7,49 18 51 N,122 51 23 W,WGS84,49.314444,-122.856389
3,Armstrong,City,1.0,82L/6,50 26 53 N,119 11 48 W,WGS84,50.448333,-119.196667
4,Ashcroft,Village (1),3.0,92I/11,50 43 16 N,121 17 00 W,WGS84,50.721389,-121.283611


In [36]:
gazeteer_filtered = gazeteer_filtered[gazeteer_filtered['Official Name'].isin(wellbeing_communities['CSD Name 2021'])]

In [37]:
#check if the gazeteer_filtered dataframe has all the communities from final_analysis_data
print(f"Total communities in final_analysis_data: {len(wellbeing_communities['CSD Name 2021'].unique())}")
print(f"Total communities in gazeteer_filtered: {len(gazeteer_filtered['Official Name'].unique())}")
missing_communities =  set(gazeteer_filtered['Official Name'].unique()) - set(wellbeing_communities['CSD Name 2021'].unique())
print(f"Missing communities in gazeteer_filtered: {len(missing_communities)}")
if missing_communities:
    print("Missing communities:")
    for community in missing_communities:
        print(f"  {community}")

Total communities in final_analysis_data: 145
Total communities in gazeteer_filtered: 145
Missing communities in gazeteer_filtered: 0


In [38]:
#find duplicates
duplicates = gazeteer_filtered[gazeteer_filtered.duplicated(subset=['Official Name'], keep=False)]
print(f"Total duplicates found: {len(duplicates)}")
if not duplicates.empty:
    print("Duplicate entries:")
    for index, row in duplicates.iterrows():
        print(f"  {row['Official Name']} (Index: {index})")


Total duplicates found: 0


In [39]:
#extract official name, feature type, datum, latitude and longitude from gazeteer_filtered
gazeteer_final = gazeteer_filtered[['Official Name', 'LatDD', 'LongDD']].copy()
gazeteer_final.rename(columns={
    'Official Name': 'Municipality_Name',
    'LatDD': 'Latitude',
    'LongDD': 'Longitude'
}, inplace=True)

#convert gazeteer_final a tidy format similar to tidy_all_data


In [40]:
import numpy as np
import pandas as pd
import os

# assume gazeteer_final already exists in the notebook
# if you need to read it instead:
# gazeteer_final = pd.read_csv(os.path.join('..','..','data','processed','gazeteer_final.csv'))

# 1) Melt to tidy format: Municipality_Name, Year, Metric, Value, Unit
tidy_gazeteer = gazeteer_final.melt(
    id_vars=['Municipality_Name'],
    value_vars=['Latitude', 'Longitude'],
    var_name='Metric',
    value_name='Value'
).copy()

# 2) Add Year (not applicable for static coordinates) and Unit
tidy_gazeteer['Year'] = 2021 # or any other placeholder year
tidy_gazeteer['Unit'] = 'decimal_degrees'

# 3) Ensure numeric values and drop any missing coordinates
tidy_gazeteer['Value'] = pd.to_numeric(tidy_gazeteer['Value'], errors='coerce')
tidy_gazeteer = tidy_gazeteer.dropna(subset=['Value']).reset_index(drop=True)

# Reorder columns to match tidy_all_data convention
tidy_gazeteer = tidy_gazeteer[['Municipality_Name', 'Year', 'Metric', 'Value', 'Unit']]

# 4) Save to processed folder
out_path = os.path.join('..','..','data','processed','gazeteer_final_tidy.csv')
tidy_gazeteer.to_csv(out_path, index=False)
print(f"Saved tidy gazeteer to: {out_path}")
tidy_gazeteer.head()

Saved tidy gazeteer to: ../../data/processed/gazeteer_final_tidy.csv


,Municipality_Name,Year,Metric,Value,Unit
0,Abbotsford,2021,Latitude,49.052222,decimal_degrees
1,Alert Bay,2021,Latitude,50.583889,decimal_degrees
2,Anmore,2021,Latitude,49.314444,decimal_degrees
3,Armstrong,2021,Latitude,50.448333,decimal_degrees
4,Ashcroft,2021,Latitude,50.721389,decimal_degrees
